# 04 — Nettoyage et contrôle qualité

Transforme les JSON bruts en Markdown propre, retire navigation et boilerplate, puis mesure la qualité du corpus.

> Vérifiez toujours les drapeaux `RUN_*` avant l'exécution. Pour cette reconstruction propre, les traitements locaux nécessaires des étapes 02 à 07 sont déjà activés ; les collectes réseau et évaluations GPU facultatives restent désactivées.


In [1]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Impossible de trouver config.yaml et src/.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Projet :', PROJECT_ROOT)
print('Python :', sys.executable)
print('Données actives :', PROJECT_ROOT / 'data')


Projet : <PROJECT_ROOT>
Python : c:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs\RAG ENSEA\.venv\Scripts\python.exe
Données actives : <PROJECT_ROOT>\data


In [2]:
import frontmatter
import pandas as pd
from config import load_config
from cleaner import HTMLCleaner

config = load_config(PROJECT_ROOT / 'config.yaml')
CLEANED_DIR = config.project.data_path / 'cleaned'
RUN_CLEANING = True
if RUN_CLEANING:
    cleaner = HTMLCleaner(config)
    cleaner.raw_dir = config.project.data_path / 'raw'
    cleaner.cleaned_dir = CLEANED_DIR
    cleaned_documents = cleaner.clean_all(clear_output=True)
    print('Documents nettoyés :', len(cleaned_documents))
print('Corpus analysé :', CLEANED_DIR)


Documents nettoyés : 1084
Corpus analysé : <PROJECT_ROOT>\data\cleaned


In [3]:
records = []
for path in sorted(CLEANED_DIR.glob('*.md')):
    document = frontmatter.load(path)
    records.append({
        **dict(document.metadata),
        'file': path.name,
        'characters': len(document.content),
        'lines': len(document.content.splitlines()),
    })
cleaned_df = pd.DataFrame(records)
summary = pd.Series({
    'documents': len(cleaned_df),
    'urls_uniques': cleaned_df['url'].nunique() if not cleaned_df.empty else 0,
    'caractères_médians': int(cleaned_df['characters'].median()) if not cleaned_df.empty else 0,
    'documents_trop_courts': int(cleaned_df['characters'].lt(config.cleaning.min_content_length).sum()) if not cleaned_df.empty else 0,
})
display(summary.to_frame('valeur'))


,valeur
documents,1084
urls_uniques,1084
caractères_médians,1730
documents_trop_courts,0


In [4]:
if not cleaned_df.empty:
    display(cleaned_df.groupby(['doc_type', 'extraction_method']).size().rename('documents').to_frame())
    display(cleaned_df.nlargest(10, 'characters')[['title', 'category', 'characters', 'url']])
    assert cleaned_df['url'].is_unique
    assert cleaned_df['characters'].ge(config.cleaning.min_content_length).all()
    print('Contrôles de structure validés.')


documents
doc_type    extraction_method                  
page        structured_main_fallback          7
            structured_main_richer          232
            trafilatura_precision           692
pdf         pdf_text                        149
spreadsheet spreadsheet_cells                 4

,title,category,characters,url
1061,la cote d ivoire en chiffres edition 2025 92,anstat,1125926,/api/source-files/ANSTAT/Statistiques_Generale...
744,ab548a85a76d5a85,wp-content,1024327,https://ensea.ed.ci/wp-content/uploads/2023/08...
888,ce9cbec15d934243,wp-content,500976,https://ensea.ed.ci/wp-content/uploads/2023/07...
58,RGPH 2014,anstat,397442,/api/source-files/ANSTAT/Recensements_RGPH/RGP...
65,RGPH 98TOME1,anstat,330062,/api/source-files/ANSTAT/Recensements_RGPH/RGP...
430,Loi de finances 2020,autres_institutions,220952,/api/source-files/AUTRES_INSTITUTIONS/Finances...
292,Loi de finances 2021,autres_institutions,206521,/api/source-files/AUTRES_INSTITUTIONS/Finances...
230,3 Rapport Economique Financier REF 2017,autres_institutions,197305,/api/source-files/AUTRES_INSTITUTIONS/Accords_...
102,RÉPERTOIRE NATIONAL DES ENTREPRISES DE CÔTE D'...,anstat,128816,/api/source-files/ANSTAT/R%C3%89PERTOIRE%20NAT...
1030,5 Memorendum Politique Economiques financieres,autres_institutions,89666,/api/source-files/AUTRES_INSTITUTIONS/Accords_...


Contrôles de structure validés.


In [5]:
markdown_list_items = 0
documents_with_lists = 0
for path in sorted(CLEANED_DIR.glob('*.md')):
    content = frontmatter.load(path).content
    item_count = sum(
        line.lstrip().startswith(('- ', '* '))
        for line in content.splitlines()
    )
    markdown_list_items += item_count
    documents_with_lists += int(item_count > 0)
display(pd.Series({
    'documents_avec_listes_markdown': documents_with_lists,
    'items_de_liste_preserves': markdown_list_items,
}).to_frame('valeur'))
assert markdown_list_items > 0, (
    "Aucune liste HTML n'a été conservée en Markdown."
)


,valeur
documents_avec_listes_markdown,478
items_de_liste_preserves,7850


## Résultat

Un document nettoyé reste une unité documentaire complète. Il conserve ses métadonnées en front-matter YAML ; il n'est pas encore découpé ni vectorisé. Les listes HTML restent des listes Markdown séparées afin de préserver leur sens.


## Conservation des faits pendant le nettoyage

Le nettoyage porte toujours sur les corpus institutionnel, économique et Facebook. Les faits
tabulaires issus des PDF économiques sont conservés comme métadonnées structurées ; ils ne sont
pas aplatis ni recalculés à partir du texte nettoyé.

In [6]:
cleaned_fact_rows = []
for path in sorted(CLEANED_DIR.glob('*.md')):
    document = frontmatter.load(path)
    for fact in document.metadata.get('table_facts', []) or []:
        cleaned_fact_rows.append({
            'file': path.name,
            'url': document.metadata.get('url', ''),
            **fact,
        })
cleaned_facts_df = pd.DataFrame(cleaned_fact_rows)
malformed_indicator_count = int((~cleaned_facts_df['indicator'].map(
    lambda value: any(char.isalpha() for char in str(value))
)).sum()) if not cleaned_facts_df.empty else 0
display(pd.Series({
    'faits_tabulaires_conservés': len(cleaned_facts_df),
    'documents_avec_faits': cleaned_facts_df['file'].nunique() if not cleaned_facts_df.empty else 0,
    'pages_distinctes': cleaned_facts_df[['url', 'page']].drop_duplicates().shape[0] if not cleaned_facts_df.empty else 0,
    'indicateurs_sans_lettres': malformed_indicator_count,
}).to_frame('valeur'))
assert not cleaned_facts_df.empty, 'Aucun fait tabulaire n’a traversé le nettoyage.'
assert malformed_indicator_count == 0, 'Des poids ou codes numériques ont été pris pour des indicateurs.'
display(cleaned_facts_df[
    cleaned_facts_df['indicator'].str.casefold().isin({'population homme', 'rapport de masculinité'})
    & cleaned_facts_df['period'].astype(str).eq('2025')
][['page', 'indicator', 'period', 'value', 'unit', 'url']])

,valeur
faits_tabulaires_conservés,53394
documents_avec_faits,51
pages_distinctes,475
indicateurs_sans_lettres,0


,page,indicator,period,value,unit,url
3495,10,Population homme,2025,17 103 895,Nombre,/api/source-files/ANSTAT/Statistiques_Generale...
3517,10,Rapport de masculinité,2025,"109,3",Nombre,/api/source-files/ANSTAT/Statistiques_Generale...
